In [6]:
import sys
sys.path.insert(0, "../..")

from ingestion.theirstack_client import TheirStackClient
from infra.snowflake_client import SnowflakeLoader

client = TheirStackClient()

In [7]:
job_ids = [
    735075886, 735321929, 735473065, 735308450, 733389679, 733942488,
    732708479, 732486522, 732681402, 732605030, 732887632, 731239725,
    731700763, 729973405, 729978552, 729466073, 729697490, 726312048,
    725056202, 724454118, 723592915, 724964814, 723592941, 724533888,
    724244447, 723564744, 724187171, 724187177, 722593402, 722684005,
    722998644, 720222233, 721617563, 723641108, 718122706, 719386066,
    723931848, 719339861, 719439128, 719584540,
]

In [8]:
batch_1 = job_ids[:25]
batch_2 = job_ids[25:]

paid_jobs = []

for batch in (batch_1, batch_2):
    result = client._paid_fetch(batch, label="Data Scientist")
    paid_jobs.extend(result)
    print(f"Batch of {len(batch)} → got {len(result)} jobs back")

print(f"Total: {len(paid_jobs)} jobs")

Batch of 25 → got 25 jobs back
Batch of 15 → got 15 jobs back
Total: 40 jobs


In [9]:
# eyeball a couple titles before committing to Snowflake
for job in paid_jobs[:5]:
    print(job.get("job_title"), "—", job.get("company_name") or job.get("company_object", {}).get("name"))

Data Scientist — Findigs, Inc.
Data Scientist, NLP (Systematic Trading) — Thurn Partners
Data Scientist — Scale.jobs
Data Scientist — Gartner
Data Scientist I, SCOT-Inbound, Planning Optimization — Amazon


In [10]:
rows = client._to_snowflake_rows(paid_jobs, label="Data Scientist")

with SnowflakeLoader() as loader:
    load_results = loader.load(rows)
    for table, count in load_results.items():
        print(f"{table}: {count} rows inserted")

RAW.THEIRSTACK.SRC_POSTINGS: 40 rows inserted
